# Objective 2.2
Predicting goals in a single match

We have a huge no.of columns at the start, which will be reduced to 8, these only look big these are around 12/13 features.

## Train/ Test split
We will be splitting at the start to prevent data leakage, grouped by match, and stratified by stage ( so every stage gets balanced distribution).

In [6]:
import pandas as pd
df = pd.read_csv("obj2.2.csv")
print(df.columns.to_list())

['match_no', 'fifa_match_id', 'kickoff_utc', 'stage', 'team', 'team_code', 'opponent', 'opp_code', 'team_match_no', 'goals', 'goals_90', 'went_to_et', 'knockout', 'stage_ord', 'venue', 'city', 'venue_country', 'fifa_points_own', 'fifa_points_opp', 'elo_own', 'elo_opp', 'alt_ascent_diff', 'squad_value_own', 'squad_value_opp', 'squad_age_own', 'squad_age_opp', 'home_own', 'home_opp', 'home_venue', 'rest_diff', 'tz_shift_diff', 'prior_gf', 'prior_ga', 'opp_prior_gf', 'opp_prior_ga', 'n_before', 'min90_before', 'sum_xg_before', 'sum_xga_before', 'opp_n_before', 'opp_min90_before', 'opp_sum_xg_before', 'opp_sum_xga_before']


In [13]:
SEED = 2026

# Sanity check
assert df.shape == (208, 43), df.shape
assert df.isna().sum().sum() == 0, df.isna().sum()[lambda s: s > 0]
assert not df.duplicated(["match_no", "team"]).any()
assert (df["team"] != df["opponent"]).all()
assert (df.groupby("match_no").size() == 2).all()
assert ((df["stage_ord"] == 0) == (df["match_no"] <= 72)).all()
assert ((df["knockout"] == 1) == (df["match_no"] > 72)).all()


matches = df.groupby("match_no", as_index=False).first()

counts = matches["knockout"].value_counts()
print(counts)


knockout
0    72
1    32
Name: count, dtype: int64


 StratifiedGroupKfold splits the matches into 5 chunks, always keeping both rows of a match together so there’s no leakage, and gives each chunk about the same group/knockout mix.

We take the first chunk, about 21 matches, as the test set, and later reuse the same tool on the training data for cross-validation.

In [15]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

sgkf = StratifiedGroupKFold(n_splits=5, random_state=SEED, shuffle=True)
train_idx, test_idx = next(sgkf.split(df, y=df['knockout'], groups=df['match_no'])) #Knockout games are smaller in counts

test_ids = np.sort(df.loc[test_idx, "match_no"].unique())
pd.DataFrame({"match_no": test_ids}).to_csv("test_match_ids.csv", index=False)

In [16]:
df["split"] = np.where(df["match_no"].isin(test_ids), "test", "train")
print(pd.crosstab(df["stage_ord"], df["split"], margins=True))
train = df[df["split"] == "train"].copy()

split      test  train  All
stage_ord                  
0            30    114  144
1             6     26   32
2             4     12   16
3             0      8    8
4             0      4    4
5             2      2    4
All          42    166  208


We will properly split after making the features count 8.